In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

from dotenv import load_dotenv


e:\Agentic AI\LangGraph\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()


llm = HuggingFaceEndpoint(
  repo_id="meta-llama/Meta-Llama-3-8B-Instruct",

  task = "text-generation"
)

model = ChatHuggingFace(llm = llm)

In [3]:
class Blogstate(TypedDict):

  title : str
  outline: str
  content: str
  

In [4]:
def create_outline(state: Blogstate)->Blogstate:

  title = state['title']
  prompt = f"generate a short and crisp outline for a blog on the topic"
  outline = model.invoke(prompt).content

  state['outline'] = outline

  return state


In [11]:
def create_blog(state: Blogstate)->Blogstate:
  title = state['title']

  outline = state['outline']
  prompt = f"write a short blog on a title using this following outline: {outline}"
  content = model.invoke(prompt).content
  state['content'] = content
  return state

In [6]:
graph = StateGraph(Blogstate)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [13]:
initial_state = {'title':'Evolution of AI in india'}
final_state= workflow.invoke(initial_state)
print('outline:')
print(final_state['outline'])
print('final blog:')
print( final_state['content'])

outline:
Here's a short and crisp outline for a blog on the topic:

**Title:** "Unlocking Productivity: Mastering Time Management for a Stress-Free Life"

**I. Introduction**

- Briefly introduce the importance of time management in today's fast-paced world
- Mention the purpose of the blog: to share practical tips on effective time management

**II. Understanding Your Time Management Goals**

- Identify personal goals and priorities
- Discuss the importance of setting realistic goals and deadlines
- Provide examples of goal-setting templates

**III. Effective Time Management Techniques**

- Introduce the Pomodoro Technique for focused work sessions
- Discuss the concept of time blocking for scheduling tasks
- Explain the importance of taking regular breaks

**IV. Avoiding Distractions and Procrastination**

- Discuss common distractions and how to minimize them
- Share strategies for overcoming procrastination
- Introduce the concept of "eating that frog" (aka tackling the least enjoy